In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. テーブルのカラム情報を確認する

In [2]:
%%sql

SHOW COLUMNS FROM customer;

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


## 2. SELECT句

#### - 1つのカラムを取得する

In [4]:
%%sql

SELECT first_name
FROM customer
LIMIT 5;

first_name
MARY
PATRICIA
LINDA
BARBARA
ELIZABETH


#### - 2つ以上のカラムを取得する (カラムを記述する順番は問わない)

In [5]:
%%sql

SELECT first_name, last_name
FROM customer
LIMIT 5;

first_name,last_name
MARY,SMITH
PATRICIA,JOHNSON
LINDA,WILLIAMS
BARBARA,JONES
ELIZABETH,BROWN


In [8]:
%%sql

-- 取得するカラムの順番は、元のテーブルの順番とは関係ない
SELECT last_name, first_name
FROM customer
LIMIT 5;

last_name,first_name
SMITH,MARY
JOHNSON,PATRICIA
WILLIAMS,LINDA
JONES,BARBARA
BROWN,ELIZABETH


#### - すべてのカラムを取得する

In [10]:
%%sql

SELECT *
FROM customer
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
4,2,BARBARA,JONES,BARBARA.JONES@sakilacustomer.org,8,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20


#### - DISTINCT演算子

- `DISTINCT`は、`SELECT`した結果から重複する行を除外し、一意の値だけを取得する演算子である。

- `SELECT句`が実行された後に適用される。

```sql
SELECT DISTINCT カラム名
FROM テーブル名;
```

```sql
SELECT DISTINCT カラム名, カラム名
FROM テーブル名;
```

from -> where -> select -> distinct -> order by -> limit

In [19]:
%%sql

SELECT DISTINCT store_id
FROM customer;

store_id
1
2


In [25]:
%%sql

SELECT
    store_id, active
FROM customer

LIMIT 5;

store_id,active
1,1
1,1
1,1
2,1
1,1


#### ※ 注意：`DISTINCT`は、各カラムの重複を個別に除外するのではなく、`SELECT`したカラムの組み合わせ全体を基準に重複を除外する。

In [22]:
%%sql

-- store_idとactiveの組み合わせが同じ行は、1行だけ取得する
SELECT DISTINCT store_id, active
FROM customer;

store_id,active
1,1
2,1
2,0
1,0


## 3. WHERE句

- 特定の行に対して比較演算や論理演算を行い、結果が`TRUE`となる行だけを絞り込む。

```sql
SELECT カラム
FROM テーブル
WHERE カラムの値に対する比較演算・論理演算;
```

#### 1) 比較演算

- 数値、文字列、日付などに使用できるが、主に数値データを持つカラムの比較に使用される。

```text
<、<=、=、>、>=、<>、!=、!<、!>
```

In [27]:
%%sql

SELECT film_id, title, length
FROM film
WHERE length <= 50
LIMIT 5;

film_id,title,length
2,ACE GOLDFINGER,48
3,ADAPTATION HOLES,50
15,ALIEN CENTER,46
83,BLUES INSTINCT,50
192,CROSSING DIVORCE,50


In [28]:
%%sql

SELECT film_id, title, length
FROM film
WHERE length = 50
LIMIT 5;

film_id,title,length
3,ADAPTATION HOLES,50
83,BLUES INSTINCT,50
192,CROSSING DIVORCE,50
524,LION UNCUT,50
607,MUPPET MILE,50


In [29]:
%%sql

SELECT film_id, title, length
FROM film
WHERE length <> 50
LIMIT 5;

film_id,title,length
1,ACADEMY DINOSAUR,86
2,ACE GOLDFINGER,48
4,AFFAIR PREJUDICE,117
5,AFRICAN EGG,130
6,AGENT TRUMAN,169


In [30]:
%%sql

-- 同じ演算
SELECT film_id, title, length
FROM film
WHERE length != 50
LIMIT 5;

film_id,title,length
1,ACADEMY DINOSAUR,86
2,ACE GOLDFINGER,48
4,AFFAIR PREJUDICE,117
5,AFRICAN EGG,130
6,AGENT TRUMAN,169


In [35]:
%%sql

-- 文字列の値にも使用できる
SELECT film_id, title, length
FROM film
WHERE length(title) <> 50
LIMIT 5;

film_id,title,length
1,ACADEMY DINOSAUR,86
2,ACE GOLDFINGER,48
3,ADAPTATION HOLES,50
4,AFFAIR PREJUDICE,117
5,AFRICAN EGG,130


#### 2) 論理演算

- `WHERE句`では、条件を組み合わせるために論理演算子（Logical Operator）を使用する。

| 論理演算子 | 意味 | 例 |
|---|---|---|
| `AND` | すべての条件が真 | `A AND B` |
| `OR` | 1つ以上の条件が真 | `A OR B` |
| `NOT` | 条件を反転する | `NOT A` |
| `IN` | 複数の値のうち、いずれかと一致する | `A IN (...)` |
| `NOT IN` | 指定した複数の値のすべてと異なる | `A NOT IN (...)` |
| `BETWEEN` | 範囲に含まれる | `A BETWEEN 1 AND 10` |
| `NOT BETWEEN` | 範囲から除外する | `A NOT BETWEEN 1 AND 10` |
| `LIKE` | 文字列のパターンを検索する | `A LIKE 'A%'` |
| `NOT LIKE` | 文字列のパターンを除外する | `A NOT LIKE 'A%'` |
| `IS NULL` | `NULL`かどうかを確認する | `A IS NULL` |
| `IS NOT NULL` | `NULL`ではないか確認する | `A IS NOT NULL` |
| `EXISTS` | サブクエリの結果が存在するか確認する | `EXISTS (...)` |

In [37]:
%%sql

SELECT *
FROM customer
WHERE active = 1
AND store_id = 1
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20
7,1,MARIA,MILLER,MARIA.MILLER@sakilacustomer.org,11,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [38]:
%%sql

SELECT *
FROM customer
WHERE store_id = 1
OR store_id = 2
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20
7,1,MARIA,MILLER,MARIA.MILLER@sakilacustomer.org,11,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [39]:
%%sql

SELECT *
FROM customer
WHERE NOT active = 1
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
16,2,SANDRA,MARTIN,SANDRA.MARTIN@sakilacustomer.org,20,0,2006-02-14 22:04:36,2006-02-15 04:57:20
64,2,JUDITH,COX,JUDITH.COX@sakilacustomer.org,68,0,2006-02-14 22:04:36,2006-02-15 04:57:20
124,1,SHEILA,WELLS,SHEILA.WELLS@sakilacustomer.org,128,0,2006-02-14 22:04:36,2006-02-15 04:57:20
169,2,ERICA,MATTHEWS,ERICA.MATTHEWS@sakilacustomer.org,173,0,2006-02-14 22:04:36,2006-02-15 04:57:20
241,2,HEIDI,LARSON,HEIDI.LARSON@sakilacustomer.org,245,0,2006-02-14 22:04:36,2006-02-15 04:57:20


In [40]:
%%sql

SELECT *
FROM customer
WHERE active <> 1
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
16,2,SANDRA,MARTIN,SANDRA.MARTIN@sakilacustomer.org,20,0,2006-02-14 22:04:36,2006-02-15 04:57:20
64,2,JUDITH,COX,JUDITH.COX@sakilacustomer.org,68,0,2006-02-14 22:04:36,2006-02-15 04:57:20
124,1,SHEILA,WELLS,SHEILA.WELLS@sakilacustomer.org,128,0,2006-02-14 22:04:36,2006-02-15 04:57:20
169,2,ERICA,MATTHEWS,ERICA.MATTHEWS@sakilacustomer.org,173,0,2006-02-14 22:04:36,2006-02-15 04:57:20
241,2,HEIDI,LARSON,HEIDI.LARSON@sakilacustomer.org,245,0,2006-02-14 22:04:36,2006-02-15 04:57:20


In [42]:
%%sql

SELECT *
FROM customer
WHERE first_name IN ('MARY', 'PATRICIA', 'LINDA')

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [43]:
%%sql

-- IN演算子を使用した場合と同じ結果
SELECT *
FROM customer
WHERE first_name = 'MARY'
   OR first_name = 'PATRICIA'
   OR first_name = 'LINDA';

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [45]:
%%sql

SELECT *
FROM customer
WHERE first_name NOT IN ('MARY', 'PATRICIA')
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
4,2,BARBARA,JONES,BARBARA.JONES@sakilacustomer.org,8,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20
6,2,JENNIFER,DAVIS,JENNIFER.DAVIS@sakilacustomer.org,10,1,2006-02-14 22:04:36,2006-02-15 04:57:20
7,1,MARIA,MILLER,MARIA.MILLER@sakilacustomer.org,11,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [47]:
%%sql

SELECT *
FROM customer
WHERE customer_id BETWEEN 10 AND 20
LIMIT 5;
-- 10を含む ～ 20を含む

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
10,1,DOROTHY,TAYLOR,DOROTHY.TAYLOR@sakilacustomer.org,14,1,2006-02-14 22:04:36,2006-02-15 04:57:20
11,2,LISA,ANDERSON,LISA.ANDERSON@sakilacustomer.org,15,1,2006-02-14 22:04:36,2006-02-15 04:57:20
12,1,NANCY,THOMAS,NANCY.THOMAS@sakilacustomer.org,16,1,2006-02-14 22:04:36,2006-02-15 04:57:20
13,2,KAREN,JACKSON,KAREN.JACKSON@sakilacustomer.org,17,1,2006-02-14 22:04:36,2006-02-15 04:57:20
14,2,BETTY,WHITE,BETTY.WHITE@sakilacustomer.org,18,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [49]:
%%sql

-- BETWEEN演算子を使用した場合と同じ結果
SELECT *
FROM customer
WHERE customer_id >= 10
  AND customer_id <= 20
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
10,1,DOROTHY,TAYLOR,DOROTHY.TAYLOR@sakilacustomer.org,14,1,2006-02-14 22:04:36,2006-02-15 04:57:20
11,2,LISA,ANDERSON,LISA.ANDERSON@sakilacustomer.org,15,1,2006-02-14 22:04:36,2006-02-15 04:57:20
12,1,NANCY,THOMAS,NANCY.THOMAS@sakilacustomer.org,16,1,2006-02-14 22:04:36,2006-02-15 04:57:20
13,2,KAREN,JACKSON,KAREN.JACKSON@sakilacustomer.org,17,1,2006-02-14 22:04:36,2006-02-15 04:57:20
14,2,BETTY,WHITE,BETTY.WHITE@sakilacustomer.org,18,1,2006-02-14 22:04:36,2006-02-15 04:57:20


In [50]:
%%sql

SELECT *
FROM customer
WHERE customer_id NOT BETWEEN 10 AND 20
LIMIT 5;

customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
4,2,BARBARA,JONES,BARBARA.JONES@sakilacustomer.org,8,1,2006-02-14 22:04:36,2006-02-15 04:57:20
5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20
